# LPM_transplant — 00_manifest_qc

**Feeds:** Fig 5n

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


            # 00 | Manifest And Metadata QC

            ## Cell Guide
            1. Resolve the workspace root and import the manifest builder.
            2. Rebuild the raw-input manifest from `data/`.
            3. Inspect input format, dimensions, z depth, and channel-order assumptions.
            

In [ ]:
import os
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
if (CWD / "scripts").exists() and (CWD / "results").exists():
    ROOT = CWD
elif (CWD.parent / "scripts").exists() and (CWD.parent / "results").exists():
    ROOT = CWD.parent.resolve()
else:
    ROOT = CWD

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Python:", sys.executable)

import pandas as pd

from scripts import build_analysis_manifest as bam

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

            ## Settings Explained

            - The current dataset contains a mixture of true Zeiss CZI files and ImageJ TIFF hyperstacks saved with a `.czi` extension.
            - This notebook writes a single manifest layer that normalizes those inputs for downstream notebooks.
            - The current biological channel assumption is fixed order:
              brightfield, DAPI, MESP2, FOXF1, donor reporter, host reporter.
            

In [ ]:
DATA_DIR = ROOT / "data"
MANIFEST_OUTPUT = ROOT / "results" / "manifests" / "raw_input_manifest.tsv"
WRITE_OUTPUTS = True

print("DATA_DIR:", DATA_DIR)
print("MANIFEST_OUTPUT:", MANIFEST_OUTPUT)
print("WRITE_OUTPUTS:", WRITE_OUTPUTS)

In [ ]:
manifest_res = bam.run_manifest_pipeline(
    root=ROOT,
    data_dir=DATA_DIR,
    output=MANIFEST_OUTPUT,
    write_output=WRITE_OUTPUTS,
)

manifest_df = manifest_res["manifest_df"].copy()
print(manifest_res["summary_text"])
display(manifest_df)

In [ ]:
format_summary = (
    manifest_df.groupby(["source_format", "size_c", "size_z"], as_index=False)
    .size()
    .sort_values(["source_format", "size_z", "size_c"])
)
display(format_summary)

channel_summary = manifest_df[["file_name", "raw_channel_names", "canonical_channel_names"]].copy()
display(channel_summary)

condition_summary = (
    manifest_df.groupby(["condition", "source_format", "size_c", "size_z"], as_index=False)
    .size()
    .sort_values(["condition", "source_format", "size_z", "size_c"])
)
display(condition_summary)